In [1]:
import pandas as pd
import torch_directml
import mlflow
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import os

u:\CODE\ReelSense\etl\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\catas\AppData\Local\Temp\ipykernel_24808\2729427373.py:4: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


In [2]:
os.environ["HF_HOME"]= "U:\\CODE\\models"
mlflow.set_experiment("ReelSense-Experiment")

device = torch_directml.device()

In [3]:
print(device)

privateuseone:0


In [4]:
from transformers import TrainerCallback

class EnhancedMLflowCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        args.logging_steps = 10 
        print("Inyectando configuración: Reportando métricas cada 10 steps...")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if "loss" in logs:
                mlflow.log_metric("train_loss", logs["loss"], step=state.global_step)
            
            if "learning_rate" in logs:
                mlflow.log_metric("learning_rate", logs["learning_rate"], step=state.global_step)
            
            if "epoch" in logs:
                mlflow.log_metric("epoch", logs["epoch"], step=state.global_step)

In [5]:
with mlflow.start_run(run_name="ReelSenseMovie_Embeddings_FineTuning"):
    mlflow.log_param("model_name", "all-MiniLM-L6-v2")
    mlflow.log_param("dataset", "Letterboxd_Movies")
    mlflow.log_param("sample_size", 20000)

    print("Loading dataset...")

    movies = pd.read_csv("../data/movies_enriched.csv").head(20000)

    columnas = [
        "name",
        "description",
        "tagline",
        "main_actors",
        "main_directors",
        "main_genres",
        "main_themes",
    ]
    movies[columnas] = movies[columnas].fillna("")

    movies["meta_texto"] = movies[
        ["name", "main_actors", "main_directors", "main_genres", "main_themes"]
    ].agg(" ".join, axis=1)
    movies["plot_texto"] = movies[["description", "tagline"]].agg(" ".join, axis=1)

    train_examples = [
        InputExample(texts=[r["meta_texto"], r["plot_texto"]])
        for _, r in movies.iterrows()
    ]
    model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)
    train_loss = losses.MultipleNegativesRankingLoss(model)
    print("Training in Radeon 7600")

    mlflowcallback = EnhancedMLflowCallback()

    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=1,
        warmup_steps=100,
        show_progress_bar=True,
        callback=mlflowcallback,
    )
    model_path = "./models/reelsense-movie-embeddings"
    model.save(model_path)
    mlflow.log_artifacts(model_path, artifact_path="model_output")

    print("Model fine-tuning completed and saved to /models/reelsense-movie-embeddings")

Loading dataset...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4120.06it/s]


Training in Radeon 7600


Step,Training Loss
500,2.014002


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.39it/s]


Model fine-tuning completed and saved to /models/reelsense-movie-embeddings


In [ ]:
#from transformers import AutoTokenizer

#enriched_movies = pd.read_csv("../data/movies_enriched.csv")

# Cargar la herramienta que cuenta los tokens
#tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

# Sacar el primer "Super Texto"
#texto_prueba = enriched_movies[['name', 'description', 'main_actors', 'main_directors', 'main_genres', 'main_themes']].iloc[0]
#texto_prueba = ' '.join(texto_prueba.astype(str))
#tokens = tokenizer.encode(texto_prueba)

#print(f"Caracteres: {len(texto_prueba)}")
#print(f"Tokens reales: {len(tokens)}")

: 